# Hour 4: Production RAG Pipeline — The Portfolio Piece

**Goal:** Build a complete Retrieval-Augmented Generation pipeline with quality gates, feedback loops, and a full audit trail.

**What you'll learn:**
- **LLM-as-judge** nodes for document grading and hallucination checking
- **Two feedback loops** — retrieval retry and generation retry
- **Audit logging** via state reducers
- **Production RAG architecture** that maps to OWASP/NIST/MITRE frameworks

**Prerequisites:** Hours 1-3 completed, Anthropic API key

**Time:** ~90 minutes

---

### Architecture Overview

Everything from Hours 1-3 converges here. Six nodes, two quality gates, two feedback loops:

```
query_analyzer → retriever → doc_grader → [relevant?]
     ↑                                        ↓ YES
     └── query_refiner ←── NO ────────────    generator → hallucination_checker → [grounded?] → END
                                                   ↑                                    ↓ NO
                                                   └────────────── retry ────────────────┘
```

### AI Security / Governance Framing

This architecture isn't just a demo — it maps to real frameworks:

| Component | Security Function | Framework Reference |
|---|---|---|
| Query Analyzer | Input transformation gate | Prompt injection detection point |
| Doc Grader | Retrieval validation | NIST AI RMF "Measure" function |
| Hallucination Checker | Output validation | OWASP LLM09 mitigation |
| Feedback Loops | Self-correction | Automated remediation |
| Checkpointing | Full audit trail | Compliance / incident review |

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

# ─── SIMULATED DOCUMENT STORE ───────────────────────────────────────────
# In production, swap for FAISS, Pinecone, ChromaDB, or any vector store.
# The retriever node interface stays the same: query in, docs out.

DOCUMENT_STORE = [
    {"id": "doc-001", "content": "LangGraph is a framework for building stateful, multi-agent applications with LLMs. It extends LangChain with directed graph orchestration, allowing developers to define complex workflows where multiple AI agents collaborate through shared state.", "source": "langgraph-docs"},
    {"id": "doc-002", "content": "State reducers in LangGraph allow multiple nodes to accumulate data in shared state fields. Using Python's Annotated type with operator.add, list fields append rather than overwrite, enabling multi-agent collaboration without data loss.", "source": "langgraph-advanced-guide"},
    {"id": "doc-003", "content": "LangGraph checkpointing saves state snapshots at every node transition. This enables pause/resume workflows, human-in-the-loop patterns, and full audit trails. Production deployments typically use PostgresSaver for persistent checkpointing.", "source": "langgraph-checkpointing-guide"},
    {"id": "doc-004", "content": "The OWASP Top 10 for LLM Applications identifies key risks including prompt injection, insecure output handling, and training data poisoning. Mitigation strategies include input validation, output filtering, and human oversight gates.", "source": "owasp-llm-top10"},
    {"id": "doc-005", "content": "The supervisor-worker pattern in LangGraph uses one LLM agent to route tasks to specialist agents. The supervisor reads the full conversation state and decides which worker should act next, enabling dynamic multi-step workflows.", "source": "langgraph-patterns"},
    {"id": "doc-006", "content": "Retrieval-Augmented Generation (RAG) grounds LLM responses in external knowledge by retrieving relevant documents before generating answers. This reduces hallucination and enables the LLM to reference authoritative sources.", "source": "rag-overview"},
    {"id": "doc-007", "content": "The best recipe for chocolate chip cookies involves creaming butter and sugar, then folding in flour and chocolate chips. Bake at 375F for 12 minutes.", "source": "cooking-blog"},
]


def simple_retriever(query: str, top_k: int = 3) -> list[dict]:
    """Keyword-based retriever. In production: vector_store.similarity_search(query, k=top_k)"""
    query_words = set(query.lower().split())
    scored = []
    for doc in DOCUMENT_STORE:
        content_words = set(doc["content"].lower().split())
        overlap = len(query_words & content_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]

print(f"Document store loaded: {len(DOCUMENT_STORE)} documents")
print(f"Test retrieval for 'LangGraph state': {[d['source'] for d in simple_retriever('LangGraph state')]}")

## State Design

The RAG state combines everything from Hours 1-3:
- **Reducers** for accumulating documents, sources, and audit logs
- **Overwrite fields** for the current query, quality decisions, and generated answer
- **A retry counter** to prevent infinite feedback loops

Notice the `audit_log` field — every node appends what it did. This gives you a complete trace of the pipeline's execution, which is critical for AI governance and debugging.

In [ ]:
class RAGState(TypedDict):
    # Input
    original_query: str                                # User's question (never changes)
    refined_query: str                                 # LLM-optimized search query (may update)

    # Retrieved context
    documents: Annotated[list[str], add]               # REDUCER: doc contents accumulate
    doc_sources: Annotated[list[str], add]             # REDUCER: citations accumulate

    # Quality gates
    docs_are_relevant: bool                            # Set by doc_grader
    answer_is_grounded: bool                           # Set by hallucination_checker

    # Output
    generation: str                                    # The generated answer

    # Control flow
    retry_count: int                                   # Loop guard
    audit_log: Annotated[list[str], add]               # REDUCER: execution trace

print("RAGState: 3 reducer fields (documents, doc_sources, audit_log)")
print("          7 overwrite fields for queries, decisions, and output")

## The Six Nodes

### Input Processing
- **Query Analyzer:** Rewrites the user's vague question into a keyword-rich search query
- **Retriever:** Searches the document store

### Quality Gate 1: Retrieval Validation
- **Doc Grader:** LLM-as-judge evaluates document relevance (PASS/FAIL)
- **Query Refiner:** If FAIL, generates a better search query for retry

### Generation + Quality Gate 2: Output Validation
- **Generator:** Synthesizes answer from documents (grounding constraint in prompt)
- **Hallucination Checker:** LLM-as-judge verifies answer is supported by evidence

In [ ]:
def query_analyzer(state: RAGState) -> dict:
    """
    NODE: query_analyzer
    Role: Optimizes the user's raw question into a retriever-friendly query.

    AI Security angle: This is an input transformation gate.
    In production, add prompt injection detection here.
    """
    query = state["original_query"]
    response = llm.invoke([
        SystemMessage(content="""You are a search query optimizer. Given a user's question,
produce a concise, keyword-rich search query that will retrieve the most relevant documents.
Respond with ONLY the optimized query, nothing else."""),
        HumanMessage(content=f"User question: {query}"),
    ])
    refined = response.content.strip()
    print(f"  [query_analyzer] '{query}' → '{refined}'")
    return {
        "refined_query": refined,
        "audit_log": [f"[query_analyzer] Refined '{query}' → '{refined}'"],
    }


def retriever(state: RAGState) -> dict:
    """
    NODE: retriever
    Role: Searches document store using the refined query.
    Writes: documents (APPENDS), doc_sources (APPENDS), audit_log (APPENDS)
    """
    query = state["refined_query"]
    results = simple_retriever(query, top_k=3)
    doc_contents = [doc["content"] for doc in results]
    doc_sources = [doc["source"] for doc in results]
    print(f"  [retriever] Found {len(results)} docs: {doc_sources}")
    return {
        "documents": doc_contents,
        "doc_sources": doc_sources,
        "audit_log": [f"[retriever] Retrieved {len(results)} docs: {', '.join(doc_sources)}"],
    }

print("Query Analyzer and Retriever nodes defined")

In [ ]:
def doc_grader(state: RAGState) -> dict:
    """
    NODE: doc_grader (LLM-as-judge)
    Role: Evaluates retrieved document relevance.

    AI Security: Retrieval validation gate.
    Prevents irrelevant context from polluting generation.
    Maps to NIST AI RMF "Measure" function.
    """
    query = state["original_query"]
    documents = state["documents"]
    doc_text = "\n\n---\n\n".join(documents[-3:])
    response = llm.invoke([
        SystemMessage(content="""You are a document relevance grader. Given a user's question
and a set of retrieved documents, determine if the documents contain information
relevant to answering the question.
Respond with ONLY "relevant" or "not_relevant". Nothing else."""),
        HumanMessage(content=f"Question: {query}\n\nDocuments:\n{doc_text}"),
    ])
    decision = response.content.strip().lower()
    is_relevant = "relevant" in decision and "not_relevant" not in decision
    print(f"  [doc_grader] {decision} → {'PASS' if is_relevant else 'FAIL'}")
    return {
        "docs_are_relevant": is_relevant,
        "audit_log": [f"[doc_grader] Relevance: {'PASS' if is_relevant else 'FAIL'}"],
    }


def generator(state: RAGState) -> dict:
    """
    NODE: generator
    Role: Synthesizes answer from documents ONLY.

    The system prompt explicitly constrains to provided context —
    a hallucination mitigation (OWASP LLM09).
    """
    query = state["original_query"]
    documents = state["documents"]
    doc_text = "\n\n---\n\n".join(documents)
    response = llm.invoke([
        SystemMessage(content="""You are a precise research assistant. Answer the user's
question using ONLY information from the provided documents. If the documents don't
contain enough information, say so explicitly. Do not add information beyond what
the documents provide. Cite which document supports each claim.
Keep your answer under 5 sentences."""),
        HumanMessage(content=f"Question: {query}\n\nDocuments:\n{doc_text}"),
    ])
    answer = response.content.strip()
    print(f"  [generator] Generated answer ({len(answer)} chars)")
    return {
        "generation": answer,
        "audit_log": [f"[generator] Produced answer ({len(answer)} chars)"],
    }

print("Doc Grader and Generator nodes defined")

In [ ]:
def hallucination_checker(state: RAGState) -> dict:
    """
    NODE: hallucination_checker (LLM-as-judge)
    Role: Verifies generated answer is grounded in source documents.

    AI Security: OUTPUT VALIDATION — the last gate before the user.
    Maps to:
    - OWASP LLM09 (Overreliance / Hallucination)
    - NIST AI RMF MG-2.2 (Monitor for unacceptable outputs)
    - MITRE ATLAS: defense against model manipulation
    """
    answer = state["generation"]
    documents = state["documents"]
    doc_text = "\n\n---\n\n".join(documents)
    response = llm.invoke([
        SystemMessage(content="""You are a fact-checking auditor. Compare the generated answer
against the source documents. Determine if every claim in the answer is supported
by information in the documents.
Respond with ONLY "grounded" or "not_grounded". Nothing else."""),
        HumanMessage(content=f"Answer to check:\n{answer}\n\nSource documents:\n{doc_text}"),
    ])
    decision = response.content.strip().lower()
    is_grounded = "grounded" in decision and "not_grounded" not in decision
    print(f"  [hallucination_checker] {decision} → {'PASS' if is_grounded else 'FAIL'}")
    return {
        "answer_is_grounded": is_grounded,
        "audit_log": [f"[hallucination_checker] Grounding: {'PASS' if is_grounded else 'FAIL'}"],
    }


def query_refiner(state: RAGState) -> dict:
    """
    NODE: query_refiner
    Role: When docs aren't relevant, tries different search keywords.
    Also bumps the retry counter (loop guard).
    """
    response = llm.invoke([
        SystemMessage(content="""The previous search query didn't return relevant results.
Generate a different search query for the same question. Try different keywords,
synonyms, or rephrasings. Respond with ONLY the new query."""),
        HumanMessage(content=f"Original question: {state['original_query']}\nFailed query: {state['refined_query']}"),
    ])
    new_query = response.content.strip()
    new_count = state.get("retry_count", 0) + 1
    print(f"  [query_refiner] Retry {new_count}: → '{new_query}'")
    return {
        "refined_query": new_query,
        "retry_count": new_count,
        "audit_log": [f"[query_refiner] Retry {new_count}: refined to '{new_query}'"],
    }

print("Hallucination Checker and Query Refiner nodes defined")

## Routers and Feedback Loops

Two conditional edges create two feedback loops:

**Loop 1 — Retrieval retry:** If the doc grader says "not relevant," the query refiner generates better search terms and routes back to the retriever. New documents accumulate via the reducer.

**Loop 2 — Generation retry:** If the hallucination checker says "not grounded," the graph routes back to the generator for another attempt with the same (good) evidence.

Both loops have **retry guards** — after a configurable number of retries, the pipeline proceeds with the best available result rather than looping forever. This is a production requirement, not just a nicety.

In [ ]:
# ─── ROUTERS ─────────────────────────────────────────────────────────────

def route_after_grading(state: RAGState) -> str:
    """After doc grading: proceed to generate, or refine and retry?"""
    if state.get("docs_are_relevant", False):
        return "generate"
    if state.get("retry_count", 0) >= 2:
        print("  [router] Max retries reached, generating with what we have")
        return "generate"
    return "refine"


def route_after_hallucination_check(state: RAGState) -> str:
    """After hallucination check: accept answer, or regenerate?"""
    if state.get("answer_is_grounded", False):
        return "accept"
    if state.get("retry_count", 0) >= 3:
        print("  [router] Max retries, accepting with warning")
        return "accept"
    return "regenerate"


# ─── WIRE THE GRAPH ─────────────────────────────────────────────────────
graph = StateGraph(RAGState)

graph.add_node("query_analyzer", query_analyzer)
graph.add_node("retriever", retriever)
graph.add_node("doc_grader", doc_grader)
graph.add_node("generator", generator)
graph.add_node("hallucination_checker", hallucination_checker)
graph.add_node("query_refiner", query_refiner)

# Linear flow
graph.add_edge(START, "query_analyzer")
graph.add_edge("query_analyzer", "retriever")
graph.add_edge("retriever", "doc_grader")

# Quality Gate 1: relevant → generate, not relevant → refine → retry
graph.add_conditional_edges("doc_grader", route_after_grading, {
    "generate": "generator",
    "refine": "query_refiner",
})
graph.add_edge("query_refiner", "retriever")  # Retrieval retry loop

# Quality Gate 2: grounded → accept, not grounded → regenerate
graph.add_edge("generator", "hallucination_checker")
graph.add_conditional_edges("hallucination_checker", route_after_hallucination_check, {
    "accept": END,
    "regenerate": "generator",  # Generation retry loop
})

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("RAG pipeline compiled: 6 nodes, 2 quality gates, 2 feedback loops")

## Running the Pipeline

Two test queries:
1. **"How does LangGraph handle state in multi-agent systems?"** — Should match well against our docs
2. **"What are the OWASP risks for LLM applications?"** — Tests relevance grading with less-targeted docs

Watch the audit log — it traces every decision the pipeline made.

In [ ]:
print("=" * 70)
print("QUERY 1: LangGraph State Management")
print("=" * 70)
print()

config = {"configurable": {"thread_id": "rag-session-1"}}
result = app.invoke({
    "original_query": "How does LangGraph handle state in multi-agent systems?",
    "refined_query": "", "documents": [], "doc_sources": [],
    "docs_are_relevant": False, "answer_is_grounded": False,
    "generation": "", "retry_count": 0, "audit_log": [],
}, config=config)

print("\n" + "=" * 70)
print("ANSWER")
print("=" * 70)
print(result["generation"])

print("\nSOURCES:", list(set(result["doc_sources"])))
print(f"\nQUALITY: Relevant={result['docs_are_relevant']}, Grounded={result['answer_is_grounded']}")

In [ ]:
print("=" * 70)
print("AUDIT LOG (full execution trace)")
print("=" * 70)
for entry in result["audit_log"]:
    print(f"  {entry}")

In [ ]:
print("=" * 70)
print("QUERY 2: OWASP LLM Risks")
print("=" * 70)
print()

config2 = {"configurable": {"thread_id": "rag-session-2"}}
result2 = app.invoke({
    "original_query": "What are the OWASP risks for LLM applications?",
    "refined_query": "", "documents": [], "doc_sources": [],
    "docs_are_relevant": False, "answer_is_grounded": False,
    "generation": "", "retry_count": 0, "audit_log": [],
}, config=config2)

print("\nANSWER:", result2["generation"])
print("\nAUDIT LOG:")
for entry in result2["audit_log"]:
    print(f"  {entry}")

## Key Takeaways — The Full Picture

### Architecture Decisions You Can Defend in an Interview

**"Why LLM-as-judge instead of rule-based grading?"**
Rules are brittle — they break on edge cases. An LLM can evaluate semantic relevance, not just keyword overlap. The trade-off is latency and cost, which is why you use it selectively (at quality gates, not every node).

**"Why two separate feedback loops?"**
Retrieval failures and generation failures have different root causes and different fixes. A bad search query needs refinement; a hallucinated answer needs regeneration with the same (good) evidence. Separating the loops makes each one targeted and debuggable.

**"Why retry guards instead of letting the LLM decide when to stop?"**
Defense in depth. The LLM *should* produce good results, but a guaranteed exit condition prevents runaway costs and latency. In production, you'd also set `recursion_limit` on the compiled graph.

**"Where does domain expertise live?"**
Not in the graph topology — that's generic orchestration. Domain expertise lives in the **SystemMessage prompts** inside the grader and checker nodes. A legal RAG system uses the same graph but different evaluation criteria. Being able to articulate this distinction is the signal.

### Framework Mapping (Resume-Ready)

| What You Built | Framework Reference |
|---|---|
| Input validation (query analyzer) | Prompt injection detection point |
| Retrieval validation (doc grader) | NIST AI RMF "Measure" function |
| Output validation (hallucination checker) | OWASP LLM09 mitigation |
| Feedback loops with guards | Automated remediation with bounded retry |
| Checkpointed audit trail | Compliance / SOC2 / incident review |
| Quality gates before user output | MITRE ATLAS defense layers |

---
*You've completed the full tutorial. You now have a working, production-pattern multi-agent RAG pipeline with quality gates, feedback loops, and a full audit trail.*